In [1]:
import os
import re
import numpy as np
import pandas as pd

from scipy.ndimage import binary_closing
from scipy.signal import butter, lfilter, lfilter_zi


# =====================================================
# settings
# =====================================================
mus = ['TA', 'RF', 'VL', 'BF']

TARGET_ROWS = 30000

folder_001 = r"C:\Users\masay\Documents\EMG_OpenSource\001\merged_data_mini"

folder_003 = r"C:\Users\masay\Documents\EMG_OpenSource\003\merged_data"

folder_006 = r"C:\Users\masay\Documents\EMG_OpenSource\006\Final_Processed_EMG"


# =====================================================
# causal bandpass filter
# =====================================================
def emg_lfilter(
    emg,
    fs=1000,
    lowcut=20,
    highcut=450,
    order=2
):

    emg = np.asarray(emg)

    nyq = fs / 2

    low = lowcut / nyq
    high = highcut / nyq

    b, a = butter(
        order,
        [low, high],
        btype='band'
    )

    zi = lfilter_zi(
        b,
        a
    ) * emg[0]

    emg_filt, _ = lfilter(
        b,
        a,
        emg,
        zi=zi
    )

    return emg_filt


# =====================================================
# causal lowpass filter
# =====================================================
def lowpass_lfilter(
    x,
    fs=1000,
    cutoff=30,
    order=2
):

    x = np.asarray(x)

    nyq = fs / 2

    norm = cutoff / nyq

    b, a = butter(
        order,
        norm,
        btype='low'
    )

    zi = lfilter_zi(
        b,
        a
    ) * x[0]

    y, _ = lfilter(
        b,
        a,
        x,
        zi=zi
    )

    return y


# =====================================================
# exponential RMS envelope
# =====================================================
def emg_envelope_exp_rms(
    emg,
    alpha=0.03
):

    env = np.zeros_like(emg)

    for i in range(len(emg)):

        if i == 0:

            env[i] = abs(
                emg[i]
            )

        else:

            env[i] = np.sqrt(

                (1 - alpha)
                * env[i - 1]**2

                +

                alpha
                * emg[i]**2
            )

    return env


# =====================================================
# causal TKEO
# psi[n] = x[n-1]^2 - x[n-2]x[n]
# =====================================================
def causal_tkeo(x):

    x = np.asarray(x)

    y = np.zeros_like(x)

    y[2:] = (

        x[1:-1] ** 2

        -

        x[:-2] * x[2:]

    )

    return y

# =====================================================
# feature-wise z-score normalization
# =====================================================
def zscore_trial(x):

    x = np.asarray(
        x,
        dtype=np.float32
    )

    mean = np.mean(
        x,
        axis=0,
        keepdims=True
    )

    std = np.std(
        x,
        axis=0,
        keepdims=True
    )

    std[std < 1e-8] = 1.0

    return (

        x

        - mean

    ) / std


# =====================================================
# EMG processing
# =====================================================
def process_emg(df):

    raw_processed = []

    rms_processed = []

    raw_dict = {}

    rms_dict = {}

    # =================================
    # each muscle
    # =================================
    for n in mus:

        raw = emg_lfilter(
            df[n].values
        )

        raw_dict[n] = raw

        rect = np.abs(
            raw
        )

        rms = emg_envelope_exp_rms(
            rect,
            alpha=0.03
        )

        rms_dict[n] = rms

        raw_processed.append(
            raw
        )

        rms_processed.append(
            rms
        )

    # =================================
    # [T,4]
    # =================================
    raw_processed = np.array(
        raw_processed
    ).T

    rms_processed = np.array(
        rms_processed
    ).T

    # =================================
    # TKEO
    # =================================
    ta_tkeo = causal_tkeo(
        raw_dict["TA"]
    )

    bf_tkeo = causal_tkeo(
        raw_dict["BF"]
    )

    ta_tkeo = np.abs(
        ta_tkeo
    )

    bf_tkeo = np.abs(
        bf_tkeo
    )

    ta_tkeo_lp = lowpass_lfilter(
        ta_tkeo,
        cutoff=30
    )

    bf_tkeo_lp = lowpass_lfilter(
        bf_tkeo,
        cutoff=30
    )

    tkeo_features = np.stack(

        [
            ta_tkeo_lp,
            bf_tkeo_lp
        ],

        axis=1

    )

    # =================================
    # RMS derivative
    # =================================
    derivative_features = []

    for n in mus:

        x = rms_dict[n]

        dx = np.zeros_like(
            x
        )

        dx[1:] = (

            x[1:]

            -

            x[:-1]

        )

        dx_lp = lowpass_lfilter(

            dx,

            cutoff=15

        )

        derivative_features.append(

            dx_lp

        )

    derivative_features = np.stack(

        derivative_features,

        axis=1

    )

    # =================================
    # feature-wise normalization
    # =================================
    raw_processed = zscore_trial(

        raw_processed

    )

    rms_processed = zscore_trial(

        rms_processed

    )

    tkeo_features = zscore_trial(

        tkeo_features

    )

    derivative_features = zscore_trial(

        derivative_features

    )

    # =================================
    # concat
    # =================================
    X = np.concatenate(

        [

            raw_processed,       # 4ch
            rms_processed,       # 4ch
            tkeo_features,       # 2ch
            derivative_features  # 4ch

        ],

        axis=1

    )

    return X


# =====================================================
# subject extraction
# =====================================================
def extract_subject_001(filename):

    m = re.search(
        r"subject(\d+)",
        filename
    )

    return int(
        m.group(1)
    )


def extract_subject_006(filename):

    m = re.search(
        r"P(\d{4})",
        filename
    )

    return int(
        m.group(1)
    )


# =====================================================
# split rule
# =====================================================
def is_train_001(s):
    return 1 <= s <= 14


def is_val_001(s):
    return 15 <= s <= 19


def is_train_003(s):
    return 1 <= s <= 109


def is_val_003(s):
    return 110 <= s <= 135


def is_train_006(s):
    return 1 <= s <= 25


def is_val_006(s):
    return 26 <= s <= 30


# =====================================================
# storage
# =====================================================
train_data = {}

val_data = {}

skipped = []


# =====================================================
# dataset 001
# =====================================================
files_001 = os.listdir(
    folder_001
)

for f in files_001:

    if not f.endswith(".csv"):
        continue

    path = os.path.join(
        folder_001,
        f
    )

    df = pd.read_csv(
        path
    )

    if len(df) != TARGET_ROWS:

        skipped.append(
            ("001", f, len(df))
        )

        continue

    subj = extract_subject_001(
        f
    )

    # =================================
    # stance
    # =================================
    stance = df["Stance"].values

    stance = binary_closing(
        stance,
        structure=np.ones(5)
    ).astype(int)

    # =================================
    # EMG
    # =================================
    X = process_emg(
        df
    )

    y = stance

    if is_train_001(subj):

        train_data.setdefault(
            ("001", subj),
            []
        ).append(
            (X, y)
        )

    if is_val_001(subj):

        val_data.setdefault(
            ("001", subj),
            []
        ).append(
            (X, y)
        )


# =====================================================
# dataset 003
# =====================================================
for i in range(1, 136):

    for j in range(1, 3):

        path = (
            fr"{folder_003}\P{i:04d}_{j:02d}_merged.csv"
        )

        if not os.path.exists(
            path
        ):
            continue

        df = pd.read_csv(
            path
        )

        if len(df) != TARGET_ROWS:

            skipped.append(

                (
                    "003",
                    os.path.basename(path),
                    len(df)
                )
            )

            continue

        # =================================
        # stance
        # =================================
        stance = df["Stance"].values

        stance = binary_closing(
            stance,
            structure=np.ones(5)
        ).astype(int)

        # =================================
        # EMG
        # =================================
        X = process_emg(
            df
        )

        y = stance

        if is_train_003(i):

            train_data.setdefault(
                ("003", i),
                []
            ).append(
                (X, y)
            )

        if is_val_003(i):

            val_data.setdefault(
                ("003", i),
                []
            ).append(
                (X, y)
            )


# =====================================================
# dataset 006
# =====================================================
files_006 = os.listdir(
    folder_006
)

for f in files_006:

    if not f.endswith(".csv"):
        continue

    path = os.path.join(
        folder_006,
        f
    )

    df = pd.read_csv(
        path
    )

    if len(df) != TARGET_ROWS:

        skipped.append(
            ("006", f, len(df))
        )

        continue

    subj = extract_subject_006(
        f
    )

    # =================================
    # stance
    # =================================
    stance = df["Stance"].values

    stance = binary_closing(
        stance,
        structure=np.ones(5)
    ).astype(int)

    # =================================
    # EMG
    # =================================
    X = process_emg(
        df
    )

    y = stance

    if is_train_006(subj):

        train_data.setdefault(
            ("006", subj),
            []
        ).append(
            (X, y)
        )

    if is_val_006(subj):

        val_data.setdefault(
            ("006", subj),
            []
        ).append(
            (X, y)
        )


# =====================================================
# pack
# =====================================================
def pack(data_dict):

    X_all = []

    y_all = []

    for key, data in data_dict.items():

        X = np.stack(
            [d[0] for d in data]
        )

        y = np.stack(
            [d[1] for d in data]
        )

        X_all.append(
            X
        )

        y_all.append(
            y
        )

    return (

        np.concatenate(
            X_all
        ),

        np.concatenate(
            y_all
        )
    )


# =====================================================
# final arrays
# =====================================================
X_train, y_train = pack(
    train_data
)

X_val, y_val = pack(
    val_data
)


# =====================================================
# info
# =====================================================
print(
    "TRAIN:",
    X_train.shape
)

print(
    "VAL:",
    X_val.shape
)

print(
    "channels:",
    X_train.shape[-1]
)

print(
    "\ntrain subjects:",
    len(train_data)
)

print(
    "val subjects:",
    len(val_data)
)

if skipped:

    print(
        "\nskipped:",
        len(skipped)
    )

TRAIN: (356, 30000, 14)
VAL: (93, 30000, 14)
channels: 14

train subjects: 148
val subjects: 36


In [2]:
np.savez(
    "opensource_emg_dataset_with_zscore.npz",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val
)